In [1]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection


In [2]:
csv_path = project_root + "/data/air_traffic_gold.csv"
con = get_ibis_connection(
    backend="duckdb",
    duckdb_csv_path=csv_path,
)


In [3]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(base_url=base_url, api_key=api_key, temperature=0, model=model)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema

distinct_values = get_character_distinct_values(
    con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
)

distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", user_template),
]

prompt_template = ChatPromptTemplate.from_messages(messages)


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()


In [ ]:
chain = prompt_template | llm


In [ ]:
def basic_sql_agent(
    chain, question, tbl_name, schema, con, chat_history, additional_context=""
):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
            "chat_history": chat_history.messages,
        }
    )
    query = llm_output.content

    chat_history.add_user_message(question)
    chat_history.add_ai_message(query)
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_" * 60)
    output = con.raw_sql(query)
    return output, chat_history


In [ ]:
con.sql(f"SELECT COUNT(*) FROM {tbl_name}").execute()

In [ ]:
question = "Please delete rows of passengers landed in terminal 1"


In [ ]:
output, chat_history = basic_sql_agent(
    chain,
    question=question,
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    chat_history=chat_history,
    con=con,
)


In [ ]:
con.sql(f"SELECT COUNT(*) FROM {tbl_name}").execute()
